In [ ]:
# Debug-fetch one contract's verified sources from Etherscan V2 (no compilation).
# - Prints a concise summary + selected fields from the payload
# - Saves:
#     ./debug_fetch/<address>/raw_payload.json          (verbatim API result[0])
#     ./debug_fetch/<address>/sourcecode_raw.txt        (verbatim 'SourceCode' string)
#     ./debug_fetch/<address>/parsed_sourcecode.json    (if 'SourceCode' parses as JSON)
#     ./debug_fetch/<address>/paths_map.json            (original path -> sanitized path)
#     ./debug_fetch/<address>/<...>.sol                 (written files, if any; or Flattened.sol if forced)
#
# Requirements: requests
# Usage: set ETHERSCAN_API_KEY in your environment, then run this cell.

import os, json, re, time, hashlib
from pathlib import Path
import requests
from dotenv import load_dotenv

API_URL = "https://api.etherscan.io/v2/api"

# ---- choose the address you want to inspect ----
address_to_debug = "0x3d41832e9d70cb232e7c14e2170116a419d8ad27"  # change me
chain_id = 1
timeout = 30.0
rate_delay = 0.25
follow_proxy = True       # if Proxy=1, try fetching its Implementation instead
force_single = True       # if JSON parse yields 0 files, force saving Flattened.sol from the raw string

# ---- helper functions ----
def fetch_v2(address: str, api_key: str, chain_id: int, timeout: float):
    params = {
        "module": "contract",
        "action": "getsourcecode",
        "address": address,
        "chainid": chain_id,
        "apikey": api_key,
    }
    r = requests.get(API_URL, params=params, timeout=timeout)
    r.raise_for_status()
    payload = r.json()
    if payload.get("status") != "1" or not payload.get("result"):
        raise RuntimeError(f"Etherscan V2 error: {payload}")
    return payload["result"][0]

def _json_load_maybe(s: str):
    try:
        return True, json.loads(s)
    except Exception:
        return False, None

def unbox_sourcecode(source_str: str):
    """
    Try very hard to turn Etherscan 'SourceCode' into a dict (standard-input JSON).
    Returns: (obj_or_None, last_seen_string)
    """
    if not isinstance(source_str, str): return None, ""
    s = source_str.strip()
    if not s: return None, ""
    last_s = s

    # A) direct load
    ok, obj = _json_load_maybe(s)
    if ok and isinstance(obj, dict): return obj, last_s
    if ok and isinstance(obj, str): last_s = obj; s = obj

    # B) unwrap one layer of braces or quotes
    if (s.startswith("{{") and s.endswith("}}")) or (s.startswith('"{') and s.endswith('}"')):
        inner = s[1:-1]
        ok, obj = _json_load_maybe(inner)
        if ok and isinstance(obj, dict): return obj, last_s
        if ok and isinstance(obj, str): last_s = obj; s = obj

    # C) iterative decoding (JSON string containing JSON string)
    for _ in range(3):
        ok, obj = _json_load_maybe(s)
        if ok and isinstance(obj, dict): return obj, last_s
        if ok and isinstance(obj, str): last_s = obj; s = obj; continue
        break

    # D) trim quotes/backticks
    t = s.strip().strip("`").strip("'").strip('"').strip()
    if t and t != s:
        ok, obj = _json_load_maybe(t)
        if ok and isinstance(obj, dict): return obj, last_s

    return None, last_s  # treat as single-file string

SAFE_CHARS_RE = re.compile(r"[^0-9A-Za-z._/\-]+")


def safe_relpath(p: str, default_name: str = "Contract.sol") -> Path:
    s = (p or "").strip().replace("\\", "/")
    s = re.sub(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", "", s)  # strip schemes
    s = re.sub(r"^[A-Za-z]:/", "", s)                   # strip win drive
    s = s.lstrip("/")                                   # drop absolute
    parts = []
    for seg in s.split("/"):
        if not seg or seg in (".", ".."): continue
        parts.append(SAFE_CHARS_RE.sub("_", seg))
    if not parts: parts = [default_name]
    rel = Path(*parts)
    if rel.suffix.lower() != ".sol":
        rel = rel / default_name
    return rel

def unique_path(base: Path, rel: Path) -> Path:
    target = base / rel
    if not target.exists(): return target
    stem, suf = target.stem, target.suffix or ".sol"
    h = hashlib.sha1(str(target).encode("utf-8")).hexdigest()[:8]
    return target.with_name(f"{stem}-{h}{suf}")

def _normalize_sources_dict(src_obj):
    # Case A: standard input JSON
    if isinstance(src_obj, dict) and isinstance(src_obj.get("sources"), dict):
        return src_obj["sources"]
    # Case B: Etherscan solc-m (top-level: filename -> {"content": "..."} )
    if isinstance(src_obj, dict) and all(isinstance(v, dict) and "content" in v for v in src_obj.values()):
        return src_obj
    return None

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def write_sources_to_disk(base_dir: Path, address: str, src_obj: dict, single_file: Optional[str], contract_name: str):
    proj_dir = base_dir / address.lower()
    _ensure_dir(proj_dir)

    # NEW: accept both standard JSON and solc-m maps
    sources = _normalize_sources_dict(src_obj) if isinstance(src_obj, dict) else None
    if sources:
        n = 0
        for raw_path, meta in sources.items():
            content = meta.get("content") if isinstance(meta, dict) else (meta if isinstance(meta, str) else "")
            if not isinstance(content, str) or not content.strip():
                continue
            fpath = proj_dir / raw_path
            _ensure_dir(fpath.parent)
            fpath.write_text(content)
            n += 1

        # Optional but handy: save a standard-JSON you can compile from later
        std = {
            "language": "Solidity",
            "sources": {k: {"content": (v["content"] if isinstance(v, dict) else v)} for k, v in sources.items()},
        }
        (proj_dir / "standard_input.json").write_text(json.dumps(std, indent=2))
        return n, proj_dir

    # Fallback: single-file
    solname = f"{contract_name or 'Contract'}.sol"
    (proj_dir / solname).write_text(single_file or "")
    return (1 if (single_file and single_file.strip()) else 0), proj_dir


# ---- main one-off debug flow ----
load_dotenv()
api_key = os.environ.get("ETHERSCAN_API_KEY")
if not api_key:
    raise SystemExit("Please set ETHERSCAN_API_KEY in your environment before running.")

addr = address_to_debug.lower().strip()
out_dir = Path("debug_fetch") / addr
out_dir.mkdir(parents=True, exist_ok=True)

print(f"[info] Fetching {addr} (chain_id={chain_id}) …")
primary = fetch_v2(addr, api_key, chain_id, timeout)

# Follow proxy (optional)
chosen = primary
impl_addr = (primary.get("Implementation") or "").strip().lower()
is_proxy = str(primary.get("Proxy", "")).strip() == "1"
if follow_proxy and is_proxy and impl_addr:
    time.sleep(rate_delay)
    print(f"[info] Contract is a proxy → fetching implementation {impl_addr}")
    try:
        chosen = fetch_v2(impl_addr, api_key, chain_id, timeout)
        (out_dir / "raw_payload.proxy.json").write_text(json.dumps(primary, indent=2))
    except Exception as e:
        print(f"[warn] Failed to fetch implementation {impl_addr}: {e}")

# Always save the exact payload and raw SourceCode
(out_dir / "raw_payload.json").write_text(json.dumps(chosen, indent=2))
src = chosen.get("SourceCode", "")
(out_dir / "sourcecode_raw.txt").write_text(src if isinstance(src, str) else str(src))

# Print a human summary
def peek(s, n=400):
    s = s if isinstance(s, str) else str(s)
    s = s.replace("\r", "\\r").replace("\n", "\\n")
    return (s[:n] + ("… (truncated)" if len(s) > n else ""))

print("\n=== Summary ===")
print("Address:            ", addr)
print("Proxy?:             ", is_proxy, " Implementation:", impl_addr or "—")
print("ContractName:       ", chosen.get("ContractName"))
print("CompilerVersion:    ", chosen.get("CompilerVersion"))
print("Language:           ", chosen.get("Language"))
print("OptimizationUsed:   ", chosen.get("OptimizationUsed"))
print("Runs:               ", chosen.get("Runs"))
print("EVMVersion:         ", chosen.get("EVMVersion"))
print("SourceCode length:  ", len(src if isinstance(src, str) else str(src)))
print("SourceCode peek:    ", peek(src, 300))

# Try unboxing and writing files
default_name = f"{(chosen.get('ContractName') or 'Contract').strip() or 'Contract'}.sol"
obj, last_str = unbox_sourcecode(src)

if isinstance(obj, dict):
    (out_dir / "parsed_sourcecode.json").write_text(json.dumps(obj, indent=2))
    # Write multi-file if possible
    n = write_sources_from_obj(out_dir, obj, default_name)
    print(f"[info] parsed SourceCode → wrote {n} file(s) to {out_dir}")
    if n == 0 and force_single and isinstance(last_str, str) and last_str.strip():
        # Force a single .sol so we can still inspect it
        f = out_dir / "Flattened.sol"
        f.write_text(last_str)
        print(f"[info] wrote Forced single file: {f}")
else:
    # Treat as single-file string if requested
    if force_single and isinstance(last_str, str) and last_str.strip():
        f = out_dir / "Flattened.sol"
        f.write_text(last_str)
        print(f"[info] wrote Forced single file: {f}")
    else:
        print("[info] Could not parse JSON and 'force_single' is off; only saved raw payload + raw string.")

print(f"\n[done] Inspect folder: {out_dir.resolve()}")


In [ ]:
import os, requests

# --- set the address you want to inspect ---
address = "0x3d41832e9d70cb232e7c14e2170116a419d8ad27"  # change me
chain_id = 1  # Ethereum mainnet

api_key = os.environ.get("ETHERSCAN_API_KEY")
if not api_key:
    raise SystemExit("Please set ETHERSCAN_API_KEY in your environment.")

params = {
    "module": "contract",
    "action": "getsourcecode",
    "address": address,
    "chainid": chain_id,
    "apikey": api_key,
}

url = "https://api.etherscan.io/v2/api"
resp = requests.get(url, params=params, timeout=30)

print("HTTP", resp.status_code)
print("URL:", resp.url)
print("\n--- RAW RESPONSE TEXT BELOW ---\n")
print(resp.text)

# (optional) save the exact response as-is:
# with open(f"etherscan_raw_{address.lower()}.json", "w") as f:
#     f.write(resp.text)


In [ ]:
import os, requests, json

addr = "0x3d41832e9d70cb232e7c14e2170116a419d8ad27"  # the one you just queried
api_key = os.environ["ETHERSCAN_API_KEY"]

r = requests.get(
    "https://api.etherscan.io/v2/api",
    params={
        "module": "contract",
        "action": "getsourcecode",
        "address": addr,
        "chainid": 1,
        "apikey": api_key,
    },
    timeout=30,
)
payload = r.json()
sc = payload["result"][0]["SourceCode"]

print("SourceCode type:", type(sc).__name__)
obj = json.loads(sc)  # <-- this is the key step: it's a JSON string
print("Has 'sources' key:", isinstance(obj, dict) and "sources" in obj)
print("Looks like solc-m map:", isinstance(obj, dict) and any(isinstance(v, dict) and "content" in v for v in obj.values()))
print("Top-level keys sample:", list(obj.keys())[:6])


In [ ]:
# Debug-fetch one contract's verified sources from Etherscan V2 (no compilation).
# Saves to: ./debug_fetch/<address>/
#   - raw_payload.json          (verbatim API result[0])
#   - sourcecode_raw.txt        (verbatim 'SourceCode' string)
#   - parsed_sourcecode.json    (if 'SourceCode' parses as JSON)
#   - paths_map.json            (original path -> saved relative path)
#   - *.sol files               (if writing is enabled)
#
# Requirements: requests
# Optional: python-dotenv (if you want to load ETHERSCAN_API_KEY from .env)

import os, json, re, time, hashlib
from typing import Optional, Dict, Tuple
from pathlib import Path

import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

API_URL = "https://api.etherscan.io/v2/api"

# ---- configure here ----
address_to_debug = "0x3d41832e9d70cb232e7c14e2170116a419d8ad27"  # change me
chain_id = 1            # Ethereum mainnet
timeout = 30.0
rate_delay = 0.25
follow_proxy = True     # if Proxy=1, also fetch Implementation
WRITE_FILES = True      # write parsed files to disk
FORCE_SINGLE = True     # if parsing yields 0 files, save Flattened.sol from raw string
OUT_ROOT = "debug_fetch"

# ---- helpers ----
def fetch_v2(address: str, api_key: str, chain_id: int, timeout: float) -> Dict:
    params = {
        "module": "contract",
        "action": "getsourcecode",
        "address": address,
        "chainid": chain_id,
        "apikey": api_key,
    }
    r = requests.get(API_URL, params=params, timeout=timeout)
    r.raise_for_status()
    payload = r.json()
    if payload.get("status") != "1" or not payload.get("result"):
        raise RuntimeError(f"Etherscan V2 error: {payload}")
    return payload["result"][0]

def _json_load_maybe(s: str):
    try:
        return True, json.loads(s)
    except Exception:
        return False, None

def unbox_sourcecode(source_str: str):
    """
    Try hard to turn Etherscan 'SourceCode' string into a dict.
    Returns (obj_or_None, last_seen_string).
    Handles standard JSON and JSON-encoded JSON. Works for 'solc-m' maps too.
    """
    if not isinstance(source_str, str): return None, ""
    s = source_str.strip()
    if not s: return None, ""
    last_s = s

    ok, obj = _json_load_maybe(s)
    if ok and isinstance(obj, dict): return obj, last_s
    if ok and isinstance(obj, str): last_s = obj; s = obj  # JSON string containing JSON

    if (s.startswith("{{") and s.endswith("}}")) or (s.startswith('"{') and s.endswith('}"')):
        inner = s[1:-1]
        ok, obj = _json_load_maybe(inner)
        if ok and isinstance(obj, dict): return obj, last_s
        if ok and isinstance(obj, str): last_s = obj; s = obj

    for _ in range(3):
        ok, obj = _json_load_maybe(s)
        if ok and isinstance(obj, dict): return obj, last_s
        if ok and isinstance(obj, str): last_s = obj; s = obj; continue
        break

    t = s.strip().strip("`").strip("'").strip('"').strip()
    if t and t != s:
        ok, obj = _json_load_maybe(t)
        if ok and isinstance(obj, dict): return obj, last_s

    return None, last_s

def _normalize_sources_dict(src_obj: dict) -> Optional[dict]:
    """
    Accept both shapes:
    A) Standard input JSON: { "language": "...", "sources": { path: {content: "..."} } }
    B) Etherscan solc-m:     { "File.sol": {"content": "..."} , ... }
    Returns a plain { path: {content: "..."} } dict or None.
    """
    if isinstance(src_obj, dict) and isinstance(src_obj.get("sources"), dict):
        return src_obj["sources"]
    if isinstance(src_obj, dict) and all(isinstance(v, dict) and "content" in v for v in src_obj.values()):
        return src_obj
    return None

SAFE_CHARS_RE = re.compile(r"[^0-9A-Za-z._/\-]+")

def safe_relpath(p: str, default_name: str = "Contract.sol") -> Path:
    s = (p or "").strip().replace("\\", "/")
    s = re.sub(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", "", s)  # strip schemes
    s = re.sub(r"^[A-Za-z]:/", "", s)                   # strip win drive
    s = s.lstrip("/")                                   # drop absolute
    parts = []
    for seg in s.split("/"):
        if not seg or seg in (".", ".."): continue
        parts.append(SAFE_CHARS_RE.sub("_", seg))
    if not parts: parts = [default_name]
    rel = Path(*parts)
    if rel.suffix.lower() != ".sol":
        rel = rel / default_name
    return rel

def unique_path(base: Path, rel: Path) -> Path:
    target = base / rel
    if not target.exists(): return target
    stem, suf = target.stem, target.suffix or ".sol"
    h = hashlib.sha1(str(target).encode("utf-8")).hexdigest()[:8]
    return target.with_name(f"{stem}-{h}{suf}")

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def write_sources_to_disk(base_dir: Path, address: str, src_obj: dict, single_file: Optional[str], contract_name: str) -> Tuple[int, Path]:
    """
    Writes normalized multi-file sources, or falls back to a single file.
    Returns (n_files_written, project_dir).
    """
    proj_dir = base_dir / address.lower()
    _ensure_dir(proj_dir)

    sources = _normalize_sources_dict(src_obj) if isinstance(src_obj, dict) else None
    if sources:
        n = 0
        paths_map = {}
        for raw_path, meta in sources.items():
            content = meta.get("content") if isinstance(meta, dict) else (meta if isinstance(meta, str) else "")
            if not isinstance(content, str) or not content.strip():
                continue
            rel = safe_relpath(str(raw_path), default_name="Contract.sol")
            fpath = unique_path(proj_dir, rel)
            _ensure_dir(fpath.parent)
            fpath.write_text(content)
            n += 1
            paths_map[str(raw_path)] = str(rel)

        # Save a standard-JSON snapshot and the path map for transparency
        std = {
            "language": "Solidity",
            "sources": {k: {"content": (v["content"] if isinstance(v, dict) else v)} for k, v in sources.items()},
        }
        (proj_dir / "standard_input.json").write_text(json.dumps(std, indent=2))
        (proj_dir / "paths_map.json").write_text(json.dumps(paths_map, indent=2))
        return n, proj_dir

    # Fallback: single-file
    solname = f"{(contract_name or 'Contract').strip() or 'Contract'}.sol"
    fpath = proj_dir / solname
    fpath.write_text(single_file or "")
    return (1 if (single_file and str(single_file).strip()) else 0), proj_dir

def peek(s, n=400):
    s = s if isinstance(s, str) else str(s)
    s = s.replace("\r", "\\r").replace("\n", "\\n")
    return (s[:n] + ("… (truncated)" if len(s) > n else ""))

# ---- main one-off debug flow ----
api_key = os.environ.get("ETHERSCAN_API_KEY")
if not api_key:
    raise SystemExit("Please set ETHERSCAN_API_KEY in your environment (or .env).")

addr = address_to_debug.lower().strip()
out_dir = Path(OUT_ROOT) / addr
_ensure_dir(out_dir)

print(f"[info] Fetching {addr} (chain_id={chain_id}) …")
primary = fetch_v2(addr, api_key, chain_id, timeout)

# Optionally follow proxy to implementation
chosen = primary
impl_addr = (primary.get("Implementation") or "").strip().lower()
is_proxy = str(primary.get("Proxy", "")).strip() == "1"
if follow_proxy and is_proxy and impl_addr:
    time.sleep(rate_delay)
    print(f"[info] Proxy detected → fetching implementation {impl_addr}")
    try:
        chosen = fetch_v2(impl_addr, api_key, chain_id, timeout)
        (out_dir / "raw_payload.proxy.json").write_text(json.dumps(primary, indent=2))
    except Exception as e:
        print(f"[warn] Failed to fetch implementation {impl_addr}: {e}")

# Always save the exact payload and raw SourceCode
(out_dir / "raw_payload.json").write_text(json.dumps(chosen, indent=2))
src = chosen.get("SourceCode", "")
(out_dir / "sourcecode_raw.txt").write_text(src if isinstance(src, str) else str(src))

# Print a concise summary (no processing needed to see these)
print("\n=== Summary ===")
print("Address:            ", addr)
print("Proxy?:             ", is_proxy, " Implementation:", impl_addr or "—")
print("ContractName:       ", chosen.get("ContractName"))
print("CompilerVersion:    ", chosen.get("CompilerVersion"))
print("CompilerType:       ", chosen.get("CompilerType"))
print("Language:           ", chosen.get("Language"))
print("OptimizationUsed:   ", chosen.get("OptimizationUsed"))
print("Runs:               ", chosen.get("Runs"))
print("EVMVersion:         ", chosen.get("EVMVersion"))
print("SourceCode length:  ", len(src if isinstance(src, str) else str(src)))
print("SourceCode peek:    ", peek(src, 300))

# Optionally write out files for inspection
if WRITE_FILES:
    obj, last_str = unbox_sourcecode(src)
    if isinstance(obj, dict):
        (out_dir / "parsed_sourcecode.json").write_text(json.dumps(obj, indent=2))
        n, _proj = write_sources_to_disk(Path(OUT_ROOT), addr, obj, last_str, (chosen.get("ContractName") or "Contract"))
        print(f"[info] parsed SourceCode → wrote {n} file(s) to {out_dir}")
        if n == 0 and FORCE_SINGLE and isinstance(last_str, str) and last_str.strip():
            f = out_dir / "Flattened.sol"
            f.write_text(last_str)
            print(f"[info] wrote Forced single file: {f}")
    else:
        if FORCE_SINGLE and isinstance(last_str, str) and last_str.strip():
            f = out_dir / "Flattened.sol"
            f.write_text(last_str)
            print(f"[info] wrote Forced single file: {f}")
        else:
            print("[info] Did not parse JSON; only saved raw payload + raw string.")

print(f"\n[done] Inspect folder: {out_dir.resolve()}")


In [ ]:
# Bulk refetch of problematic contracts (no compilation) from Etherscan V2.
# For each address in INPUT_CSV, saves to ./debug_fetch_bulk/<address>/:
#   - raw_payload.primary.json   (verbatim primary API result[0])
#   - raw_payload.json           (verbatim chosen payload; impl for proxies)
#   - sourcecode_raw.txt         (verbatim 'SourceCode' string)
#   - parsed_sourcecode.json     (if parsable JSON)
#   - standard_input.json        (normalized standard JSON snapshot)
#   - paths_map.json             (original path -> saved relative path)
#   - *.sol files                (written per file; or Flattened.sol if forced)
#
# Summary index: ./debug_fetch_bulk_index.csv

import os, json, re, time, hashlib, csv
from typing import Optional, Dict, Tuple, List
from pathlib import Path

import pandas as pd
import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

API_URL = "https://api.etherscan.io/v2/api"

# ====== CONFIGURE THESE ======
INPUT_CSV      = "refetch_zero_files.csv"     # <-- set to your failed list CSV (or "refetch_zero_files.csv")
ADDR_COLUMN    = None                  # If None, auto-detects 'contract_address' or 'address'
OUT_ROOT       = "debug_fetch_bulk"
SUMMARY_CSV    = "debug_fetch_bulk_index.csv"

CHAIN_ID       = 1
TIMEOUT        = 30.0
RATE_DELAY     = 0.25
FOLLOW_PROXY   = True
FORCE_SINGLE   = True                  # if no files parsed, save Flattened.sol from raw string
SKIP_EXISTING  = True                  # skip address if OUT_ROOT/<addr> already exists
# =============================

api_key = os.environ.get("ETHERSCAN_API_KEY")
if not api_key:
    raise SystemExit("Please set ETHERSCAN_API_KEY in your environment (or .env).")

# ---------- helpers ----------
def fetch_v2(address: str, api_key: str, chain_id: int, timeout: float) -> Dict:
    params = {"module": "contract", "action": "getsourcecode", "address": address, "chainid": chain_id, "apikey": api_key}
    r = requests.get(API_URL, params=params, timeout=timeout)
    r.raise_for_status()
    payload = r.json()
    if payload.get("status") != "1" or not payload.get("result"):
        raise RuntimeError(f"Etherscan V2 error for {address}: {payload}")
    return payload["result"][0]

def _json_load_maybe(s: str):
    try:
        return True, json.loads(s)
    except Exception:
        return False, None

def unbox_sourcecode(source_str: str):
    """Try to turn 'SourceCode' string into a dict (supports JSON-encoded JSON)."""
    if not isinstance(source_str, str): return None, ""
    s = source_str.strip()
    if not s: return None, ""
    last_s = s

    ok, obj = _json_load_maybe(s)
    if ok and isinstance(obj, dict): return obj, last_s
    if ok and isinstance(obj, str): last_s = obj; s = obj

    if (s.startswith("{{") and s.endswith("}}")) or (s.startswith('"{') and s.endswith('}"')):
        inner = s[1:-1]
        ok, obj = _json_load_maybe(inner)
        if ok and isinstance(obj, dict): return obj, last_s
        if ok and isinstance(obj, str): last_s = obj; s = obj

    for _ in range(3):
        ok, obj = _json_load_maybe(s)
        if ok and isinstance(obj, dict): return obj, last_s
        if ok and isinstance(obj, str): last_s = obj; s = obj; continue
        break

    t = s.strip().strip("`").strip("'").strip('"').strip()
    if t and t != s:
        ok, obj = _json_load_maybe(t)
        if ok and isinstance(obj, dict): return obj, last_s

    return None, last_s

def _normalize_sources_dict(src_obj: dict) -> Optional[dict]:
    """
    Accept both shapes:
    A) Standard input JSON: { "language": "...", "sources": { path: {content: "..."} } }
    B) Etherscan solc-m:     { "File.sol": {"content": "..."} , ... }
    """
    if isinstance(src_obj, dict) and isinstance(src_obj.get("sources"), dict):
        return src_obj["sources"]
    if isinstance(src_obj, dict) and all(isinstance(v, dict) and "content" in v for v in src_obj.values()):
        return src_obj
    return None

SAFE_CHARS_RE = re.compile(r"[^0-9A-Za-z._/\-]+")

def safe_relpath(p: str, default_name: str = "Contract.sol") -> Path:
    s = (p or "").strip().replace("\\", "/")
    s = re.sub(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", "", s)  # strip schemes
    s = re.sub(r"^[A-Za-z]:/", "", s)                   # strip win drive
    s = s.lstrip("/")                                   # drop absolute
    parts = []
    for seg in s.split("/"):
        if not seg or seg in (".", ".."): continue
        parts.append(SAFE_CHARS_RE.sub("_", seg))
    if not parts: parts = [default_name]
    rel = Path(*parts)
    if rel.suffix.lower() != ".sol":
        rel = rel / default_name
    return rel

def unique_path(base: Path, rel: Path) -> Path:
    target = base / rel
    if not target.exists(): return target
    stem, suf = target.stem, target.suffix or ".sol"
    h = hashlib.sha1(str(target).encode("utf-8")).hexdigest()[:8]
    return target.with_name(f"{stem}-{h}{suf}")

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def write_sources_to_disk(base_dir: Path, address: str, src_obj: dict, single_file: Optional[str], contract_name: str) -> Tuple[int, Path]:
    """Writes normalized multi-file sources, or falls back to a single file."""
    proj_dir = base_dir / address.lower()
    _ensure_dir(proj_dir)

    sources = _normalize_sources_dict(src_obj) if isinstance(src_obj, dict) else None
    if sources:
        n = 0
        paths_map = {}
        for raw_path, meta in sources.items():
            content = meta.get("content") if isinstance(meta, dict) else (meta if isinstance(meta, str) else "")
            if not isinstance(content, str) or not content.strip():
                continue
            rel = safe_relpath(str(raw_path), default_name="Contract.sol")
            fpath = unique_path(proj_dir, rel)
            _ensure_dir(fpath.parent)
            fpath.write_text(content)
            n += 1
            paths_map[str(raw_path)] = str(rel)

        std = {
            "language": "Solidity",
            "sources": {k: {"content": (v["content"] if isinstance(v, dict) else v)} for k, v in sources.items()},
        }
        (proj_dir / "standard_input.json").write_text(json.dumps(std, indent=2))
        (proj_dir / "paths_map.json").write_text(json.dumps(paths_map, indent=2))
        return n, proj_dir

    solname = f"{(contract_name or 'Contract').strip() or 'Contract'}.sol"
    fpath = proj_dir / solname
    fpath.write_text(single_file or "")
    return (1 if (single_file and str(single_file).strip()) else 0), proj_dir

def peek(s, n=200):
    s = s if isinstance(s, str) else str(s)
    s = s.replace("\r", "\\r").replace("\n", "\\n")
    return (s[:n] + ("… (truncated)" if len(s) > n else ""))

# ---------- load addresses ----------
df = pd.read_csv(INPUT_CSV)
if ADDR_COLUMN is None:
    for c in ["contract_address", "address", "addr"]:
        if c in df.columns:
            ADDR_COLUMN = c
            break
if ADDR_COLUMN is None:
    raise ValueError(f"Could not find an address column in {INPUT_CSV} (looked for 'contract_address'/'address'/'addr').")

addrs: List[str] = (
    df[ADDR_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .loc[lambda s: s.str.startswith("0x") & (s.str.len() == 42)]
    .drop_duplicates()
    .tolist()
)

print(f"[info] Loaded {len(addrs)} unique addresses from {INPUT_CSV}")

# ---------- iterate & fetch ----------
out_root = Path(OUT_ROOT)
_ensure_dir(out_root)
rows = []

iterator = addrs
if _HAS_TQDM:
    iterator = tqdm(addrs, desc="Refetching", unit="addr")

for addr in iterator:
    try:
        out_dir = out_root / addr
        if SKIP_EXISTING and out_dir.exists() and any(out_dir.glob("*.sol")):
            rows.append({
                "contract_address": addr, "proxy": None, "implementation": None,
                "compiler_version": None, "compiler_type": None, "language": None,
                "n_files": len(list(out_dir.glob("**/*.sol"))), "saved_dir": str(out_dir),
                "status": "skipped_existing", "error": ""
            })
            continue

        time.sleep(RATE_DELAY)
        primary = fetch_v2(addr, api_key, CHAIN_ID, TIMEOUT)

        _ensure_dir(out_dir)
        (out_dir / "raw_payload.primary.json").write_text(json.dumps(primary, indent=2))

        chosen = primary
        impl_addr = (primary.get("Implementation") or "").strip().lower()
        is_proxy = str(primary.get("Proxy", "")).strip() == "1"

        if FOLLOW_PROXY and is_proxy and impl_addr:
            time.sleep(RATE_DELAY)
            try:
                chosen = fetch_v2(impl_addr, api_key, CHAIN_ID, TIMEOUT)
            except Exception as e:
                # if impl fetch fails, continue with primary
                chosen = primary

        # Always save verbatim chosen payload and raw SourceCode
        (out_dir / "raw_payload.json").write_text(json.dumps(chosen, indent=2))
        src = chosen.get("SourceCode", "")
        (out_dir / "sourcecode_raw.txt").write_text(src if isinstance(src, str) else str(src))

        # Try to parse and write
        n_files = 0
        err = ""
        try:
            obj, last_str = unbox_sourcecode(src)
            if isinstance(obj, dict):
                (out_dir / "parsed_sourcecode.json").write_text(json.dumps(obj, indent=2))
                n_files, _ = write_sources_to_disk(out_root, addr, obj, last_str, (chosen.get("ContractName") or "Contract"))
                if n_files == 0 and FORCE_SINGLE and isinstance(last_str, str) and last_str.strip():
                    (out_dir / "Flattened.sol").write_text(last_str)
                    n_files = 1
            else:
                if FORCE_SINGLE and isinstance(last_str, str) and last_str.strip():
                    (out_dir / "Flattened.sol").write_text(last_str)
                    n_files = 1
        except Exception as e:
            err = f"parse/write error: {e}"
            if FORCE_SINGLE and isinstance(src, str) and src.strip():
                (out_dir / "Flattened.sol").write_text(src)
                n_files = 1

        rows.append({
            "contract_address": addr,
            "proxy": is_proxy,
            "implementation": impl_addr or "",
            "compiler_version": chosen.get("CompilerVersion"),
            "compiler_type": chosen.get("CompilerType"),
            "language": chosen.get("Language"),
            "n_files": n_files,
            "saved_dir": str(out_dir),
            "status": "ok" if n_files > 0 and not err else ("ok_with_flattened" if n_files > 0 else "no_files"),
            "error": err,
            "sourcecode_len": len(src if isinstance(src, str) else str(src)),
            "sourcecode_peek": peek(src, 200),
        })

    except Exception as e:
        rows.append({
            "contract_address": addr, "proxy": None, "implementation": None,
            "compiler_version": None, "compiler_type": None, "language": None,
            "n_files": 0, "saved_dir": str(out_root / addr),
            "status": "error", "error": str(e),
            "sourcecode_len": None, "sourcecode_peek": "",
        })

# ---------- write summary ----------
summary_df = pd.DataFrame(rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"[done] Wrote summary: {SUMMARY_CSV}")
print(f"[done] Output root:  {out_root.resolve()}")


In [ ]:
# Recompile Solidity projects found under ./contracts based on compiled_results.csv
# - Prefers ./contracts/<addr>/standard_input.json when present (compile_standard)
# - Else compiles all .sol files in the project together (compile_standard)
# - Falls back to single-file compile only if needed
# - Uses compiler_version from CSV if present; otherwise guesses from pragma
#
# Outputs:
#   recompile_results.csv          (per-address recompile outcome)
#   compiled_results_merged.csv    (original CSV + recompile_* columns)

import os, re, json, time, traceback
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd

from solcx import (
    install_solc,
    set_solc_version,
    compile_source,
    compile_standard,
    get_installed_solc_versions,
)

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- config ----------------
INPUT_CSV   = "disl_subset100.csv"
PROJECTS_DIR = Path("contracts")
OUT_RECOMPILE = "compilation_results_final.csv"
RECOMPILE_ALL = True  # False = only rows where compile_ok != True
TIMEOUT_PER_ADDR_S = 0   # set >0 to sleep between addresses (e.g., 0.05)

# -------------- helpers -----------------
_VERSION_RE = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
_SEMVER_RE  = re.compile(r"(\d+)\.(\d+)\.(\d+)")

def _collect_sources_from_dir(project_dir: Path) -> Dict[str, str]:
    """Return {relative_path: content} for all .sol under project_dir."""
    sources: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        rel = p.relative_to(project_dir).as_posix()
        try:
            txt = p.read_text()
        except Exception:
            txt = ""
        sources[rel] = txt
    return sources

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if f.exists():
        try:
            return json.loads(f.read_text())
        except Exception:
            return None
    return None

def _pick_version_from_csv_or_pragma(csv_version: str, sources: Dict[str, str]) -> Optional[str]:
    """
    If csv_version is non-empty, return its semver (strip leading 'v').
    Else try to guess from pragmas, preferring the highest version mentioned.
    For simple ranges like '>=0.7.0 <0.9.0', pick a reasonable modern within range.
    """
    # 1) Use CSV version if present
    if isinstance(csv_version, str) and csv_version.strip():
        m = _SEMVER_RE.search(csv_version)
        if m:
            return ".".join(m.groups())

    # 2) Gather versions from pragmas
    versions: List[Tuple[int,int,int]] = []
    upper_bound_minor: Optional[int] = None
    for content in sources.values():
        for m in _VERSION_RE.finditer(content):
            expr = m.group(1).strip()
            # Try fixed or caret versions first (^0.x.y or 0.x.y)
            m2 = _SEMVER_RE.search(expr)
            if m2:
                versions.append(tuple(map(int, m2.groups())))
            # Try to detect an upper bound like "<0.9.0"
            ub = re.search(r"<\s*0\.(\d+)\.0", expr)
            if ub:
                try:
                    ubm = int(ub.group(1))
                    upper_bound_minor = ubm if (upper_bound_minor is None or ubm < upper_bound_minor) else upper_bound_minor
                except:
                    pass

    # If we saw explicit versions, take the highest by tuple
    if versions:
        v = max(versions)
        return f"{v[0]}.{v[1]}.{v[2]}"

    # If only an upper bound like <0.9.0 was seen, pick a common latest within range
    if upper_bound_minor is not None:
        # choose a sensible patch for the upper bound - 1
        # (<0.9.0) -> choose 0.8.21 ; (<0.8.0) -> 0.7.6 ; etc.
        mapping = {
            9:  "0.8.21",
            8:  "0.7.6",
            7:  "0.6.12",
            6:  "0.5.17",
            5:  "0.4.26",
        }
        return mapping.get(upper_bound_minor, "0.8.21")

    # Last resort default
    return "0.8.21"

def _ensure_solc(version: str) -> Optional[str]:
    """Install (if needed) and set the requested solc version. Return error string or None on success."""
    try:
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed:
            install_solc(version)
        set_solc_version(version)
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _compile_project(project_dir: Path, csv_version: str) -> Tuple[bool, str, str, int, str]:
    """
    Return: (ok, message, used_version, n_sources, mode)
      mode in {"standard", "all_files", "single_file", "none"}
    """
    std = _load_standard_input(project_dir)
    if std and isinstance(std, dict) and isinstance(std.get("sources"), dict) and std["sources"]:
        sources = {k: (v.get("content") if isinstance(v, dict) else str(v)) for k, v in std["sources"].items()}
        used_version = _pick_version_from_csv_or_pragma(csv_version, sources)
        err = _ensure_solc(used_version)
        if err:
            return False, err, used_version, len(sources), "standard"
        try:
            # ensure optimizer defaults if not present
            settings = std.get("settings") or {}
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {"language": "Solidity", "sources": {k: {"content": v} for k, v in sources.items()}, "settings": settings}
            _ = compile_standard(payload)
            return True, "Compiled OK", used_version, len(sources), "standard"
        except Exception as e:
            return False, f"Compilation failed: {e}", used_version, len(sources), "standard"

    # No standard_input.json: gather all files
    sources = _collect_sources_from_dir(project_dir)
    if sources:
        used_version = _pick_version_from_csv_or_pragma(csv_version, sources)
        err = _ensure_solc(used_version)
        if err:
            return False, err, used_version, len(sources), "all_files"
        try:
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": v} for k, v in sources.items()},
                "settings": {"optimizer": {"enabled": True, "runs": 200}},
            }
            _ = compile_standard(payload)
            return True, "Compiled OK", used_version, len(sources), "all_files"
        except Exception as e:
            # As a last resort, try single-file on the "main-looking" file
            try:
                # pick the file with the most content as a heuristic
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", used_version, len(sources), "single_file"
            except Exception as e2:
                return False, f"Compilation failed: {e2}", used_version, len(sources), "all_files"

    return False, "No .sol sources found", "", 0, "none"

# ---------------- main ------------------
df = pd.read_csv(INPUT_CSV)

# choose which rows to process
if RECOMPILE_ALL:
    todo = df.copy()
else:
    todo = df.loc[df.get("compile_ok") != True].copy()

rows = []
it = range(len(todo))
if _HAS_TQDM:
    it = tqdm(it, desc="Recompiling", unit="addr")

for i in it:
    row = todo.iloc[i]
    addr = str(row["contract_address"]).strip().lower()
    csv_version = str(row.get("compiler_version") or "").strip()
    proj_dir = PROJECTS_DIR / addr

    start = time.perf_counter()

    if not proj_dir.exists():
        rows.append({
            "contract_address": addr,
            "project_dir": str(proj_dir),
            "recompile_ok": False,
            "recompile_message": "Project directory not found",
            "recompile_solc": "",
            "recompile_n_sources": 0,
            "recompile_mode": "none",
            "elapsed_s": 0.0,
            "original_idx": row.get("original_idx"),
            "label": row.get("label"),
        })
        continue

    ok, msg, used_version, n_sources, mode = _compile_project(proj_dir, csv_version)
    elapsed = time.perf_counter() - start

    rows.append({
        "contract_address": addr,
        "project_dir": str(proj_dir),
        "recompile_ok": bool(ok),
        "recompile_message": str(msg),
        "recompile_solc": used_version,
        "recompile_n_sources": int(n_sources),
        "recompile_mode": mode,
        "elapsed_s": round(elapsed, 3),
        "original_idx": row.get("original_idx"),
        "label": row.get("label"),
    })

re_df = pd.DataFrame(rows)
re_df.to_csv(OUT_RECOMPILE, index=False)




print(f"[OK] Wrote: {OUT_RECOMPILE}")


In [10]:
# Re-run failed contracts with fixed exact-pragma handling + robust SolcError formatting

import re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List
from collections import Counter

import pandas as pd
from solcx import (
    install_solc,
    set_solc_version,
    compile_source,
    compile_standard,
    get_installed_solc_versions,
    get_installable_solc_versions,
)
from solcx.exceptions import SolcError

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------- config ----------
FAILED_INPUT = "compilation_results_with_pragma_fallback.csv"  # <-- change if yours is named differently
PROJECTS_DIR = Path("contracts")
LOG_DIR = Path("compile_logs")
OUT_CSV = "compilation_results_with_diagnostics.csv"
ONLY_ADDRESSES_WITH_SOURCES = True

LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- regex helpers ----------
PRAGMA_RE     = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
SEMVER_RE     = re.compile(r"\b(\d+)\.(\d+)\.(\d+)\b")
LT_UPPER_RE   = re.compile(r"<\s*(\d+)\.(\d+)\.(\d+)")
LE_UPPER_RE   = re.compile(r"<=\s*(\d+)\.(\d+)\.(\d+)")
GE_LOWER_RE   = re.compile(r">=\s*(\d+)\.(\d+)\.(\d+)")
GT_LOWER_RE   = re.compile(r">\s*(\d+)\.(\d+)\.(\d+)")
CARET_RE      = re.compile(r"\^\s*(\d+)\.(\d+)\.(\d+)")
TILDE_RE      = re.compile(r"~\s*(\d+)\.(\d+)\.(\d+)")
EXACT_ONLY_RE = re.compile(r"^\s*(\d+)\.(\d+)\.(\d+)\s*$")  # e.g., "0.8.7" with no operators

def _semver_tuple(s: str) -> Optional[Tuple[int,int,int]]:
    m = SEMVER_RE.search(s)
    return tuple(map(int, m.groups())) if m else None

def _collect_sources(project_dir: Path) -> Dict[str, str]:
    out: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        try:
            out[p.relative_to(project_dir).as_posix()] = p.read_text()
        except Exception:
            out[p.relative_to(project_dir).as_posix()] = ""
    return out

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if f.exists():
        try:
            return json.loads(f.read_text())
        except Exception:
            return None
    return None

def _installable_versions_str() -> List[str]:
    try:
        return [str(v) for v in get_installable_solc_versions()]
    except Exception:
        # safe baseline list
        return ["0.4.26","0.5.17","0.6.12","0.7.6","0.8.0","0.8.4","0.8.7","0.8.10","0.8.13","0.8.16","0.8.18","0.8.20","0.8.21","0.8.30"]

def _ensure_solc(version: str) -> Optional[str]:
    try:
        if version not in [str(v) for v in get_installed_solc_versions()]:
            install_solc(version)
        set_solc_version(version)
        return None
    except Exception as e:
        return f"Failed to set/install solc {version}: {e}"

def _infer_version_from_sources(sources: Dict[str,str]) -> Tuple[Optional[str], str]:
    """
    Prefer exact pragma (e.g., `pragma solidity 0.8.7;` → exactly 0.8.7).
    Then caret/tilde, then bounded ranges, then a sane default.
    Returns (version_str, explanation).
    """
    exacts, carets, tildes, lowers, lowers_strict, uppers, uppers_le = [], [], [], [], [], [], []

    for content in sources.values():
        for m in PRAGMA_RE.finditer(content or ""):
            expr = m.group(1).strip()
            # exact?
            em = EXACT_ONLY_RE.match(expr)
            if em:
                exacts.append(tuple(map(int, em.groups())))
                continue
            # caret / tilde
            cm = re.search(r"\^\s*(\d+)\.(\d+)\.(\d+)", expr)
            if cm:
                carets.append(tuple(map(int, cm.groups())))
            tm = re.search(r"~\s*(\d+)\.(\d+)\.(\d+)", expr)
            if tm:
                tildes.append(tuple(map(int, tm.groups())))
            # lower/upper bounds
            ge = GE_LOWER_RE.findall(expr)
            gt = GT_LOWER_RE.findall(expr)
            le = LE_UPPER_RE.findall(expr)
            lt = LT_UPPER_RE.findall(expr)
            lowers += [tuple(map(int, v)) for v in ge]
            lowers_strict += [tuple(map(int, v)) for v in gt]
            uppers_le += [tuple(map(int, v)) for v in le]
            uppers += [tuple(map(int, v)) for v in lt]

    inst = _installable_versions_str()
    note = []

    # 1) exact pragmas → use that exact version
    if exacts:
        counts = Counter(exacts)
        # prefer the most common; break ties by highest
        exact = sorted(counts.items(), key=lambda kv: (kv[1], kv[0]))[-1][0]
        vstr = f"{exact[0]}.{exact[1]}.{exact[2]}"
        if vstr in inst:
            note.append(f"exact pragma={vstr}")
            return vstr, "; ".join(note)
        # if not exactly installable, pick nearest *patch* that equals the exact tuple (usually present)
        note.append(f"exact pragma chose {vstr} (not in installable list?)")
        return vstr, "; ".join(note)

    # 2) caret: choose latest patch within the same minor of the highest caret
    if carets:
        vx = max(carets)
        note.append(f"caret={vx}")
        minor = [v for v in inst if v.startswith(f"{vx[0]}.{vx[1]}.")]
        if minor:
            return sorted(minor, key=lambda s: _semver_tuple(s))[-1], "; ".join(note)

    # 3) tilde: choose latest patch within that minor
    if tildes:
        vx = max(tildes)
        note.append(f"tilde={vx}")
        minor = [v for v in inst if v.startswith(f"{vx[0]}.{vx[1]}.")]
        if minor:
            return sorted(minor, key=lambda s: _semver_tuple(s))[-1], "; ".join(note)

    # 4) bounded ranges: honor both sides if possible
    lower_candidates = lowers + lowers_strict
    upper_candidates = uppers + uppers_le
    lower = max(lower_candidates) if lower_candidates else None
    upper = min(upper_candidates) if upper_candidates else None
    if lower or upper:
        if lower: note.append(f"lower>~={lower}")
        if upper: note.append(f"upper<~={upper}")
        def within(t):
            ok = True
            if lower and not (t >= lower): ok = False
            if upper and not (t <= upper): ok = False
            return ok
        cands = [v for v in inst if _semver_tuple(v) and within(_semver_tuple(v))]
        if cands:
            return sorted(cands, key=lambda s: _semver_tuple(s))[-1], "; ".join(note)

    # 5) default fallbacks
    for default in ["0.8.30","0.8.21","0.8.20","0.8.18","0.7.6","0.6.12","0.5.17","0.4.26"]:
        if default in inst:
            note.append(f"default={default}")
            return default, "; ".join(note)

    return (inst[-1] if inst else None), "; ".join(note)

def _safe_attr(obj, *names):
    for n in names:
        if hasattr(obj, n):
            return getattr(obj, n)
    return ""

def _tail(s: Optional[str], n=1200) -> str:
    if not s:
        return ""
    s = str(s)
    return s if len(s) <= n else ("…" + s[-n:])

def _format_error(tag: str, version: str, mode: str, e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = _safe_attr(e, "stderr_data", "stderr")
        stdout = _safe_attr(e, "stdout_data", "stdout")
        cmd    = _safe_attr(e, "command", "cmd")
        rc     = _safe_attr(e, "return_code", "code")
        return (
            f"{tag}: SolcError (v{version}, mode={mode}) rc={rc}\n"
            f"cmd: {cmd}\n"
            f"--- stderr (tail) ---\n{_tail(stderr)}\n"
            f"--- stdout (tail) ---\n{_tail(stdout)}"
        )
    return f"{tag}: {type(e).__name__}: {e}"

def _compile_with(version: str, std: Optional[dict], sources: Dict[str,str]) -> Tuple[bool,str,str,str]:
    """(ok, message, mode, main_file)"""
    err = _ensure_solc(version)
    if err:
        return False, err, "setup", ""
    # standard input first
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = std.get("settings") or {}
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {
                "language": std.get("language", "Solidity"),
                "sources": {k: {"content": (v.get("content") if isinstance(v, dict) else str(v))} for k, v in std["sources"].items()},
                "settings": settings,
            }
            _ = compile_standard(payload)
            return True, "Compiled OK", "standard", ""
        except Exception as e:
            return False, _format_error("standard", version, "standard", e), "standard", ""

    # all files together
    if sources:
        try:
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": v} for k, v in sources.items()},
                "settings": {"optimizer": {"enabled": True, "runs": 200}},
            }
            _ = compile_standard(payload)
            return True, "Compiled OK", "all_files", ""
        except Exception as e_all:
            # single-file (largest) as last resort
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file", main_rel
            except Exception as e_single:
                joined = _format_error("all_files", version, "all_files", e_all) + "\n---\n" + _format_error("single_file", version, "single_file", e_single)
                return False, joined, "all_files", ""
    return False, "No .sol sources found", "none", ""

def _nearest_patches(base_version: str, limit=4) -> List[str]:
    inst = _installable_versions_str()
    t = _semver_tuple(base_version or "")
    if not t: return []
    prefix = f"{t[0]}.{t[1]}."
    cands = [v for v in inst if v.startswith(prefix)]
    return sorted(cands, key=lambda s: _semver_tuple(s), reverse=True)[:limit]

def compile_with_diagnostics(addr: str, csv_version: str) -> dict:
    proj_dir = PROJECTS_DIR / addr
    rec = {
        "contract_address": addr,
        "project_dir": str(proj_dir),
        "ok": False,
        "message": "",
        "used_version": "",
        "strategy": "",
        "mode": "",
        "main_file": "",
        "elapsed_s": 0.0,
    }
    t0 = time.perf_counter()
    lines: List[str] = []

    if not proj_dir.exists():
        rec["message"] = "Project directory not found"
        return rec

    sources = _collect_sources(proj_dir)
    if ONLY_ADDRESSES_WITH_SOURCES and not sources:
        rec["message"] = "No .sol sources in project directory"
        return rec

    std = _load_standard_input(proj_dir)
    inferred, why = _infer_version_from_sources(sources)
    lines.append(f"[infer] {addr}: inferred={inferred} ({why})")

    # initial version: CSV semver if present, else inferred
    csv_m = SEMVER_RE.search(csv_version or "")
    first = ".".join(csv_m.groups()) if csv_m else (inferred or "0.8.30")
    lines.append(f"[initial] trying {first} (from {'CSV' if csv_m else 'inferred'})")

    tried = set()

    def attempt(ver: str, tag: str):
        ok, msg, mode, main = _compile_with(ver, std, sources)
        tried.add(ver)
        if ok:
            rec.update({"ok": True, "message": msg, "used_version": ver, "strategy": tag, "mode": mode, "main_file": main})
            lines.append(f"[{tag}] SUCCESS with {ver} mode={mode}")
        else:
            lines.append(f"[{tag}] FAIL with {ver}\n{msg}")
        return ok

    # Attempt 1
    if not attempt(first, "initial"):
        # Attempt 2: inferred (if different)
        if inferred and inferred not in tried:
            if attempt(inferred, "pragma_choice"):
                pass
        # Attempt 3: sweep nearby patches within the chosen minor (helps for caret/tilde)
        if not rec["ok"]:
            base = inferred or first
            for cand in _nearest_patches(base, limit=5):
                if cand in tried: continue
                if attempt(cand, "patch_sweep"):
                    break

    rec["elapsed_s"] = round(time.perf_counter() - t0, 3)
    if not rec["ok"] and not rec["message"]:
        rec["message"] = "Compilation failed (see log)"

    # write per-address log
    (LOG_DIR / f"{addr}.txt").write_text("\n".join(lines) + "\n")
    return rec

# -------- run on prior failures only --------
prev = pd.read_csv(FAILED_INPUT)
if "ok" not in prev.columns:
    raise SystemExit("FAILED_INPUT must have an 'ok' column from the previous run.")
fails = prev.loc[prev["ok"] == False].copy()
print(f"Re-attempting {len(fails)} failed addresses…")

rows = []
it = range(len(fails))
if _HAS_TQDM:
    it = tqdm(it, desc="Diagnostics", unit="addr")

for i in it:
    addr = str(fails.iloc[i]["contract_address"]).strip().lower()
    csv_ver = str(
        fails.iloc[i].get("used_version")
        or fails.iloc[i].get("recompile_solc")
        or fails.iloc[i].get("compiler_version")
        or ""
    )
    r = compile_with_diagnostics(addr, csv_ver)
    r["original_idx"] = fails.iloc[i].get("original_idx")
    r["label"] = fails.iloc[i].get("label")
    rows.append(r)

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print(f"[OK] wrote {OUT_CSV}")
print(f"Logs → {LOG_DIR}/<address>.txt")


Re-attempting 29 failed addresses…


Diagnostics: 100%|██████████| 29/29 [00:44<00:00,  1.53s/addr]

[OK] wrote compilation_results_with_diagnostics.csv
Logs → compile_logs/<address>.txt


In [11]:
# Recompile Solidity projects under ./contracts using a resilient strategy:
# 1) Try compiler_version from CSV if present.
# 2) If that fails, infer the HIGHEST pragma version seen across project files and try that.
# 3) If we still get a "requires different compiler version" SolcError, extract the requested version from the error and try once more.
#
# Inputs:
#   - INPUT_CSV: a CSV with columns [original_idx, label, contract_address, compiler_version, ...]
#   - contracts/<address>/ holds sources (and optionally standard_input.json)
#
# Outputs:
#   - OUT_RECOMPILE: per-address results with chosen version and attempt that worked
#
# Requirements:
#   pip install pandas py-solc-x tqdm

import re, json, time, traceback
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd

from solcx import (
    install_solc,
    set_solc_version,
    compile_source,
    compile_standard,
    get_installed_solc_versions,
)

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- config ----------------
INPUT_CSV     = "disl_subset100.csv"   # <-- change if needed
PROJECTS_DIR  = Path("contracts")
OUT_RECOMPILE = "compilation_results_final.csv"
RECOMPILE_ALL = True     # False → only rows where compile_ok != True in INPUT_CSV
SLEEP_BETWEEN = 0.0      # seconds between addresses (0 to disable)

# -------------- helpers -----------------
_VERSION_RE = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
_SEMVER_RE  = re.compile(r"(\d+)\.(\d+)\.(\d+)")
# Error line e.g.: "pragma solidity 0.8.7;" or with caret/range; we just capture a semver
_ERR_PRAGMA_VERSION_RE = re.compile(r"pragma\s+solidity[^0-9]*?(\d+\.\d+\.\d+)", re.IGNORECASE)

_current_solc: Optional[str] = None  # avoid redundant set_solc_version calls

def _collect_sources_from_dir(project_dir: Path) -> Dict[str, str]:
    """Return {relative_path: content} for all .sol under project_dir."""
    sources: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        rel = p.relative_to(project_dir).as_posix()
        try:
            txt = p.read_text()
        except Exception:
            txt = ""
        sources[rel] = txt
    return sources

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if f.exists():
        try:
            return json.loads(f.read_text())
        except Exception:
            return None
    return None

def _pick_highest_version_from_pragmas(sources: Dict[str, str]) -> Optional[str]:
    """Find all semvers in pragma lines and pick the highest X.Y.Z tuple."""
    versions: List[Tuple[int,int,int]] = []
    for content in sources.values():
        for m in _VERSION_RE.finditer(content):
            expr = m.group(1)
            # Pull any X.Y.Z present in the pragma expression (works for ^, ranges, exact)
            for m2 in _SEMVER_RE.finditer(expr):
                versions.append(tuple(map(int, m2.groups())))
    if versions:
        v = max(versions)
        return f"{v[0]}.{v[1]}.{v[2]}"
    return None

def _pick_version_from_csv_or_pragmas(csv_version: str, sources: Dict[str, str]) -> str:
    """Prefer CSV version when present; else highest pragma; else fall back to a modern default."""
    # 1) CSV
    if isinstance(csv_version, str) and csv_version.strip():
        m = _SEMVER_RE.search(csv_version)
        if m:
            return ".".join(m.groups())
    # 2) Highest pragma
    v = _pick_highest_version_from_pragmas(sources)
    if v:
        return v
    # 3) Safe default
    return "0.8.21"

def _ensure_solc(version: str) -> Optional[str]:
    """Install (if needed) and set the requested solc version. Return error string or None on success."""
    global _current_solc
    try:
        if _current_solc == version:
            return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed:
            install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _compile_with(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool, str, str]:
    """
    Attempt a compilation using `version`.
    If std is present (standard_input.json), prefer it; else compile all files together; finally try single-file.
    Returns: (ok, message, mode)
    """
    err = _ensure_solc(version)
    if err:
        return False, err, "init"

    # 1) standard_input.json path
    if std and isinstance(std, dict) and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = std.get("settings") or {}
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": (v.get("content") if isinstance(v, dict) else str(v))}
                            for k, v in std["sources"].items()},
                "settings": settings,
            }
            _ = compile_standard(payload)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, _summarize_solc_error(e, version, "standard"), "standard"

    # 2) all-files path
    if sources:
        try:
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": v} for k, v in sources.items()},
                "settings": {"optimizer": {"enabled": True, "runs": 200}},
            }
            _ = compile_standard(payload)
            return True, "Compiled OK", "all_files"
        except Exception as e_all:
            # 3) single-file fallback on the largest file
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                joined = _merge_error_summaries(
                    _summarize_solc_error(e_all, version, "all_files"),
                    _summarize_solc_error(e_single, version, "single_file"),
                )
                return False, joined, "all_files"

    return False, "No .sol sources found", "none"

def _summarize_solc_error(e: Exception, version: str, mode: str) -> str:
    # py-solc-x raises SolcError with .stderr_data/.stdout_data in some versions; be defensive
    etype = type(e).__name__
    base = f"{etype} (v{version}, mode={mode})"
    stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
    stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
    if not (stderr or stdout):
        return f"{base}: {str(e)}"
    def tail(s: str, n=1200):
        return s[-n:] if isinstance(s, str) else ""
    return f"{base}\n--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"

def _merge_error_summaries(a: str, b: str) -> str:
    return (a or "").rstrip() + ("\n---\n" if a and b else "") + (b or "")

def _extract_version_from_error(msg: str) -> Optional[str]:
    # Look inside an error summary for a pragma line that includes X.Y.Z
    m = _ERR_PRAGMA_VERSION_RE.search(msg or "")
    return m.group(1) if m else None

def _compile_project(project_dir: Path, csv_version: str) -> Tuple[bool, str, str, int, str]:
    """
    Returns: (ok, message, used_version, n_sources, mode)
    """
    std = _load_standard_input(project_dir)
    sources = None if std else _collect_sources_from_dir(project_dir)
    n_sources = (len(std["sources"]) if std and "sources" in std else (len(sources) if sources else 0))

    # First guess
    first_version = _pick_version_from_csv_or_pragmas(csv_version, (sources or
                             {k: (v.get("content") if isinstance(v, dict) else str(v)) for k, v in std["sources"].items()} if std else {}))
    ok, msg, mode = _compile_with(first_version, std, sources)
    if ok:
        return True, f"Compiled OK (initial v{first_version}, mode={mode})", first_version, n_sources, mode

    # Second attempt: if initial wasn't from pragma or might be wrong, try the highest pragma explicitly
    pragma_max = _pick_highest_version_from_pragmas(
        {k: (v.get("content") if isinstance(v, dict) else str(v)) for k, v in std["sources"].items()} if std else (sources or {})
    )
    if pragma_max and pragma_max != first_version:
        ok2, msg2, mode2 = _compile_with(pragma_max, std, sources)
        if ok2:
            return True, f"Compiled OK (pragma_max v{pragma_max}, mode={mode2})", pragma_max, n_sources, mode2
        msg = _merge_error_summaries(msg, msg2)

    # Third attempt: if compiler complained about a specific version, honor it
    hinted = _extract_version_from_error(msg)
    if hinted and hinted not in {first_version, pragma_max}:
        ok3, msg3, mode3 = _compile_with(hinted, std, sources)
        if ok3:
            return True, f"Compiled OK (hinted v{hinted}, mode={mode3})", hinted, n_sources, mode3
        msg = _merge_error_summaries(msg, msg3)

    return False, msg or "Compilation failed (no further detail)", first_version, n_sources, "none"

# ---------------- main ------------------
df = pd.read_csv(INPUT_CSV)

if RECOMPILE_ALL:
    todo = df.copy()
else:
    # Only those not compiled OK previously (handles bool, str "True", etc.)
    ok_mask = df.get("compile_ok").astype(str).str.lower().eq("true")
    todo = df.loc[~ok_mask].copy()

rows = []
iterator = range(len(todo))
if _HAS_TQDM:
    iterator = tqdm(iterator, desc="Recompiling (smart)", unit="addr")

for i in iterator:
    row = todo.iloc[i]
    addr = str(row["contract_address"]).strip().lower()
    csv_version = str(row.get("compiler_version") or "").strip()
    proj_dir = PROJECTS_DIR / addr

    t0 = time.perf_counter()

    if not proj_dir.exists():
        rows.append({
            "contract_address": addr,
            "project_dir": str(proj_dir),
            "recompile_ok": False,
            "recompile_message": "Project directory not found",
            "used_version": "",
            "n_sources": 0,
            "mode": "none",
            "elapsed_s": round(time.perf_counter() - t0, 3),
            "original_idx": row.get("original_idx"),
            "label": row.get("label"),
        })
        continue

    ok, msg, used_version, n_sources, mode = _compile_project(proj_dir, csv_version)

    rows.append({
        "contract_address": addr,
        "project_dir": str(proj_dir),
        "recompile_ok": bool(ok),
        "recompile_message": str(msg),
        "used_version": used_version,
        "n_sources": int(n_sources),
        "mode": mode,
        "elapsed_s": round(time.perf_counter() - t0, 3),
        "original_idx": row.get("original_idx"),
        "label": row.get("label"),
    })

    if SLEEP_BETWEEN > 0:
        time.sleep(SLEEP_BETWEEN)

re_df = pd.DataFrame(rows)
re_df.to_csv(OUT_RECOMPILE, index=False)
print(f"[OK] Wrote: {OUT_RECOMPILE}  ({len(re_df)} rows)")


Recompiling (smart): 100%|██████████| 100/100 [01:07<00:00,  1.48addr/s]

[OK] Wrote: compilation_results_final.csv  (100 rows)


In [12]:
# Smart (full-dataset) Solidity recompilation with robust pragma/range handling.
# - Uses CSV version if present.
# - Else intersects all pragma ranges across the project, and picks the highest valid version UNDER any upper bound.
# - If the compiler error shows a pragma line, treat '^x.y.z' as "latest in x.y.*" (not exactly x.y.z).
# - If we hit "Invalid EVM version requested", retry once with evmVersion stripped.
# - Tries: standard_input.json -> all_files -> single_file (largest file).
#
# Inputs:
#   compiled_results_dedup.csv (or change INPUT_CSV)
# Requires: pandas, py-solc-x, tqdm (optional)

import re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd
from solcx import (
    install_solc, set_solc_version, compile_source, compile_standard, get_installed_solc_versions,
)

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- config ----------------
INPUT_CSV     = "disl_subset100.csv"   # change if needed
PROJECTS_DIR  = Path("contracts")
OUT_RECOMPILE = "compilation_results_final.csv"
RECOMPILE_ALL = True
SLEEP_BETWEEN = 0.0

# Known "latest" patches per minor (stable, widely available in py-solc-x)
LATEST_PATCH = {
    (0,4): 26,
    (0,5): 17,
    (0,6): 12,
    (0,7): 6,
    (0,8): 30,  # bump if you know a newer one is available locally
}

SEMVER3   = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA    = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER    = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")

# Also read operator when we extract from error text
ERR_LINE  = re.compile(r"pragma\s+solidity\s+([^\n;]+);", re.IGNORECASE)

_current_solc: Optional[str] = None

def vtup(s: str) -> Tuple[int,int,int]:
    m = SEMVER3.search(s)
    if not m: return (0,0,0)
    return tuple(map(int, m.groups()))

def vstr(t: Tuple[int,int,int]) -> str:
    return f"{t[0]}.{t[1]}.{t[2]}"

def vmax(a: Tuple[int,int,int], b: Tuple[int,int,int]) -> Tuple[int,int,int]:
    return a if a > b else b

def vmin(a: Tuple[int,int,int], b: Tuple[int,int,int]) -> Tuple[int,int,int]:
    return a if a < b else b

def dec_patch(t: Tuple[int,int,int]) -> Tuple[int,int,int]:
    (M,m,p) = t
    if p > 0: return (M,m,p-1)
    if m > 0:
        pmax = LATEST_PATCH.get((M, m-1), 0)
        return (M, m-1, pmax if pmax>0 else 0)
    return (M,0,0)

def latest_in_minor(M: int, m: int) -> Tuple[int,int,int]:
    p = LATEST_PATCH.get((M,m))
    if not p:
        # fall back reasonably
        p = 30 if (M,m)==(0,8) else 12 if (M,m)==(0,6) else 6 if (M,m)==(0,7) else 26 if (M,m)==(0,4) else 17
    return (M,m,p)

def parse_pragma_expr(expr: str) -> Tuple[Tuple[int,int,int], Tuple[int,int,int]]:
    """
    Parse a single pragma expression like:
      "^0.8.7", ">=0.6.0 <0.8.0", "0.4.24", "~0.5.4", ">=0.8.0", "<0.9.0"
    Return an interval [lo, hi) (hi is exclusive). If no info, return (0.0.0, +inf).
    """
    lo = (0,0,0)
    hi = (99,99,99)  # sentinel "infinity"
    tokens = OP_VER.findall(expr)
    if not tokens:
        return lo, hi
    for op, a, b, c in tokens:
        vt = (int(a), int(b), int(c))
        if op in ("", None):  # exact
            lo = vmax(lo, vt)
            hi = vmin(hi, (vt[0], vt[1], vt[2]+1))  # [v, v+ε)
        elif op == "^":
            # ^M.m.p  ->  >=M.m.p  and  <M.(m+1).0
            lo = vmax(lo, vt)
            hi = vmin(hi, (vt[0], vt[1]+1, 0))
        elif op == "~":
            # ~M.m.p  ->  >=M.m.p  and  <M.(m+1).0
            lo = vmax(lo, vt)
            hi = vmin(hi, (vt[0], vt[1]+1, 0))
        elif op == ">=":
            lo = vmax(lo, vt)
        elif op == ">":
            lo = vmax(lo, (vt[0], vt[1], vt[2]+1))
        elif op == "<=":
            hi = vmin(hi, (vt[0], vt[1], vt[2]+1))
        elif op == "<":
            hi = vmin(hi, vt)
    return lo, hi

def intersect(a: Tuple[Tuple[int,int,int], Tuple[int,int,int]],
              b: Tuple[Tuple[int,int,int], Tuple[int,int,int]]) -> Tuple[Tuple[int,int,int], Tuple[int,int,int]]:
    return (vmax(a[0], b[0]), vmin(a[1], b[1]))

def pick_from_range(lo: Tuple[int,int,int], hi: Tuple[int,int,int]) -> Optional[Tuple[int,int,int]]:
    """
    Choose the highest version strictly < hi and >= lo.
    Strategy: try the 'latest' patch of the highest allowed minor; if >=hi, decrement.
    """
    # start candidate: just below hi
    cand = dec_patch(hi)
    # bump to latest patch in cand's minor
    cand = latest_in_minor(cand[0], cand[1])
    # walk down until >= lo
    while cand >= hi or cand < lo:
        if cand < lo:
            # make sure lower bound's minor is honored
            cand = latest_in_minor(lo[0], lo[1])
            if cand < lo:
                cand = (lo[0], lo[1], lo[2])
        else:
            cand = dec_patch(cand)
        if cand < lo:
            break
    return cand if (lo <= cand < hi) else None

def highest_pragma_version(sources: Dict[str,str]) -> Optional[str]:
    """
    Intersect all pragma ranges across files and pick a version.
    """
    have_any = False
    rng = ((0,0,0), (99,99,99))
    for content in sources.values():
        for pm in PRAGMA.finditer(content):
            have_any = True
            lo, hi = parse_pragma_expr(pm.group(1))
            rng = intersect(rng, (lo, hi))
    if not have_any:
        return None
    chosen = pick_from_range(*rng)
    return vstr(chosen) if chosen else None

def _collect_sources(project_dir: Path) -> Dict[str,str]:
    out: Dict[str,str] = {}
    for p in project_dir.rglob("*.sol"):
        try:
            out[p.relative_to(project_dir).as_posix()] = p.read_text()
        except Exception:
            out[p.relative_to(project_dir).as_posix()] = ""
    return out

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if f.exists():
        try:
            return json.loads(f.read_text())
        except Exception:
            return None
    return None

def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    try:
        if _current_solc == version:
            return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed:
            install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _compile_standard_resilient(payload: dict, version: str):
    try:
        return compile_standard(payload)
    except Exception as e1:
        # Retry once without evmVersion if requested invalid
        msg = (getattr(e1, "stderr_data", "") or "") + (getattr(e1, "stdout_data", "") or "") + str(e1)
        if "Invalid EVM version requested" in msg:
            st = payload.get("settings") or {}
            if "evmVersion" in st:
                st = dict(st)
                st.pop("evmVersion", None)
                payload2 = dict(payload)
                payload2["settings"] = st
                return compile_standard(payload2)  # may still raise; let it bubble
        raise

def _summarize(e: Exception, version: str, mode: str) -> str:
    et = type(e).__name__
    stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
    stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
    msg = f"{et} (v{version}, mode={mode})"
    if stderr or stdout:
        def tail(s): return s[-1200:] if isinstance(s, str) else ""
        msg += f"\n--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    else:
        msg += f": {e}"
    return msg

def _compile_with(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err:
        return False, err, "init"

    # standard_input.json route
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = dict(std.get("settings") or {})
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": (v.get("content") if isinstance(v, dict) else str(v))}
                            for k, v in std["sources"].items()},
                "settings": settings,
            }
            _ = _compile_standard_resilient(payload, version)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, _summarize(e, version, "standard"), "standard"

    # all-files route
    if sources:
        try:
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": v} for k, v in sources.items()},
                "settings": {"optimizer": {"enabled": True, "runs": 200}},
            }
            _ = _compile_standard_resilient(payload, version)
            return True, "Compiled OK", "all_files"
        except Exception as e_all:
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                return False, _summarize(e_all, version, "all_files") + "\n---\n" + _summarize(e_single, version, "single_file"), "all_files"

    return False, "No .sol sources found", "none"

def _versions_from_error_hinted_range(err_text: str, sources: Dict[str,str]) -> Optional[str]:
    """
    If compiler shows a pragma line in the error, interpret it as a RANGE (handle ^, ~, <, <=, etc.)
    and choose the best version under that upper bound (not exact).
    """
    m = ERR_LINE.search(err_text or "")
    if not m:
        return None
    lo, hi = parse_pragma_expr(m.group(1))
    chosen = pick_from_range(lo, hi)
    return vstr(chosen) if chosen else highest_pragma_version(sources)

def pick_initial_version(csv_version: str, sources: Dict[str,str]) -> str:
    # 1) CSV semver if present
    if isinstance(csv_version, str) and csv_version.strip():
        m = SEMVER3.search(csv_version)
        if m:
            return ".".join(m.groups())
    # 2) Intersect pragma ranges → highest under upper bound
    v = highest_pragma_version(sources)
    if v:
        return v
    # 3) default
    return "0.8.21"

def compile_project(project_dir: Path, csv_version: str) -> Tuple[bool,str,str,int,str]:
    std = _load_standard_input(project_dir)
    sources = None if std else _collect_sources(project_dir)
    n_sources = (len(std["sources"]) if std and "sources" in std else (len(sources) if sources else 0))

    src_for_infer = {k:(v.get("content") if isinstance(v, dict) else str(v)) for k,v in std["sources"].items()} if std else (sources or {})
    first = pick_initial_version(csv_version, src_for_infer)
    ok, msg, mode = _compile_with(first, std, sources)
    if ok: return True, f"Compiled OK (initial v{first}, mode={mode})", first, n_sources, mode

    # second try: explicit best-from-pragmas (if different from first)
    pragma_best = highest_pragma_version(src_for_infer)
    if pragma_best and pragma_best != first:
        ok2, msg2, mode2 = _compile_with(pragma_best, std, sources)
        if ok2: return True, f"Compiled OK (pragma_best v{pragma_best}, mode={mode2})", pragma_best, n_sources, mode2
        msg += "\n---\n" + msg2

    # third try: read pragma line from compiler error and pick *range*-aware version
    hinted = _versions_from_error_hinted_range(msg, src_for_infer)
    if hinted and hinted not in {first, pragma_best}:
        ok3, msg3, mode3 = _compile_with(hinted, std, sources)
        if ok3: return True, f"Compiled OK (hinted v{hinted}, mode={mode3})", hinted, n_sources, mode3
        msg += "\n---\n" + msg3

    return False, (msg or "Compilation failed (no further detail)"), first, n_sources, "none"

# ---------------- main ------------------
df = pd.read_csv(INPUT_CSV)
todo = df.copy() if RECOMPILE_ALL else df.loc[df.get("compile_ok").astype(str).str.lower() != "true"].copy()

rows = []
it = range(len(todo))
if _HAS_TQDM:
    it = tqdm(it, desc="Recompiling (smart ranges)", unit="addr")

for i in it:
    row = todo.iloc[i]
    addr = str(row["contract_address"]).strip().lower()
    proj = PROJECTS_DIR / addr
    t0 = time.perf_counter()

    if not proj.exists():
        rows.append({
            "contract_address": addr,
            "project_dir": str(proj),
            "recompile_ok": False,
            "recompile_message": "Project directory not found",
            "used_version": "",
            "n_sources": 0,
            "mode": "none",
            "elapsed_s": round(time.perf_counter() - t0, 3),
            "original_idx": row.get("original_idx"),
            "label": row.get("label"),
        })
        continue

    csv_ver = str(row.get("compiler_version") or "").strip()
    ok, msg, used, nsrc, mode = compile_project(proj, csv_ver)

    rows.append({
        "contract_address": addr,
        "project_dir": str(proj),
        "recompile_ok": bool(ok),
        "recompile_message": str(msg),
        "used_version": used,
        "n_sources": int(nsrc),
        "mode": mode,
        "elapsed_s": round(time.perf_counter() - t0, 3),
        "original_idx": row.get("original_idx"),
        "label": row.get("label"),
    })

    if SLEEP_BETWEEN > 0: time.sleep(SLEEP_BETWEEN)

out = pd.DataFrame(rows)
out.to_csv(OUT_RECOMPILE, index=False)
print(f"[OK] Wrote: {OUT_RECOMPILE} ({len(out)} rows)")


Recompiling (smart ranges): 100%|██████████| 100/100 [00:45<00:00,  2.20addr/s]

[OK] Wrote: compilation_results_final.csv (100 rows)


In [15]:
# Full-dataset Solidity recompilation with extra heuristics:
# - Range-aware pragma intersection (handles ^, ~, <, <=, >=, >)
# - Strips any SPDX lines before compile (avoids "Invalid SPDX license identifier")
# - Detects legacy fallback `function () external` and forces a 0.5.x compiler if needed
# - Ignores nonsense CSV versions (e.g., 0.9.0, 99.99.17) and falls back to pragma
# - Retries once without evmVersion on "Invalid EVM version requested"
#
# Inputs:
#   compiled_results_dedup.csv   (change INPUT_CSV if needed)
# Requires: pandas, py-solc-x, tqdm (optional)

import re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd
from solcx import (
    install_solc, set_solc_version, compile_source, compile_standard, get_installed_solc_versions,
)

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- config ----------------
INPUT_CSV     = "disl_subset200.csv"   # <-- change to your input file if needed
PROJECTS_DIR  = Path("contracts")
OUT_RECOMPILE = "compilation_results_final.csv"
RECOMPILE_ALL = True
SLEEP_BETWEEN = 0.0

# Known latest patch per minor supported by py-solc-x (adjust if you have more)
LATEST_PATCH = {
    (0,4): 26,
    (0,5): 17,
    (0,6): 12,
    (0,7): 6,
    (0,8): 30,
}

VALID_MINORS = {4,5,6,7,8}

SEMVER3   = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA    = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER    = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")
ERR_LINE  = re.compile(r"pragma\s+solidity\s+([^\n;]+);", re.IGNORECASE)

LEGACY_FALLBACK_RE = re.compile(r"function\s*\(\)\s*(external|public)", re.IGNORECASE)
SPDX_LINE_RE       = re.compile(r"^\s*//\s*SPDX-License-Identifier:.*$", re.IGNORECASE | re.MULTILINE)

_current_solc: Optional[str] = None

def vtup(s: str) -> Tuple[int,int,int]:
    m = SEMVER3.search(s)
    if not m: return (0,0,0)
    return tuple(map(int, m.groups()))

def vstr(t: Tuple[int,int,int]) -> str:
    return f"{t[0]}.{t[1]}.{t[2]}"

def vmax(a: Tuple[int,int,int], b: Tuple[int,int,int]) -> Tuple[int,int,int]:
    return a if a > b else b

def vmin(a: Tuple[int,int,int], b: Tuple[int,int,int]) -> Tuple[int,int,int]:
    return a if a < b else b

def dec_patch(t: Tuple[int,int,int]) -> Tuple[int,int,int]:
    (M,m,p) = t
    if p > 0: return (M,m,p-1)
    if m > 0:
        pmax = LATEST_PATCH.get((M, m-1), 0)
        return (M, m-1, pmax if pmax>0 else 0)
    return (M,0,0)

def latest_in_minor(M: int, m: int) -> Tuple[int,int,int]:
    p = LATEST_PATCH.get((M,m))
    if not p:
        p = 30 if (M,m)==(0,8) else 12 if (M,m)==(0,6) else 6 if (M,m)==(0,7) else 26 if (M,m)==(0,4) else 17
    return (M,m,p)

def parse_pragma_expr(expr: str) -> Tuple[Tuple[int,int,int], Tuple[int,int,int]]:
    lo = (0,0,0)
    hi = (99,99,99)
    tokens = OP_VER.findall(expr)
    if not tokens:
        return lo, hi
    for op, a, b, c in tokens:
        vt = (int(a), int(b), int(c))
        if op in ("", None):  # exact
            lo = vmax(lo, vt)
            hi = vmin(hi, (vt[0], vt[1], vt[2]+1))
        elif op == "^":
            lo = vmax(lo, vt)
            hi = vmin(hi, (vt[0], vt[1]+1, 0))
        elif op == "~":
            lo = vmax(lo, vt)
            hi = vmin(hi, (vt[0], vt[1]+1, 0))
        elif op == ">=":
            lo = vmax(lo, vt)
        elif op == ">":
            lo = vmax(lo, (vt[0], vt[1], vt[2]+1))
        elif op == "<=":
            hi = vmin(hi, (vt[0], vt[1], vt[2]+1))
        elif op == "<":
            hi = vmin(hi, vt)
    return lo, hi

def intersect(a, b):
    return (vmax(a[0], b[0]), vmin(a[1], b[1]))

def pick_from_range(lo: Tuple[int,int,int], hi: Tuple[int,int,int]) -> Optional[Tuple[int,int,int]]:
    cand = dec_patch(hi)
    cand = latest_in_minor(cand[0], cand[1])
    while cand >= hi or cand < lo:
        if cand < lo:
            cand = latest_in_minor(lo[0], lo[1])
            if cand < lo:
                cand = (lo[0], lo[1], lo[2])
        else:
            cand = dec_patch(cand)
        if cand < lo:
            break
    return cand if (lo <= cand < hi) else None

def highest_pragma_version(sources: Dict[str,str]) -> Optional[str]:
    have_any = False
    rng = ((0,0,0), (99,99,99))
    for content in sources.values():
        for pm in PRAGMA.finditer(content):
            have_any = True
            lo, hi = parse_pragma_expr(pm.group(1))
            rng = intersect(rng, (lo, hi))
    if not have_any:
        return None
    chosen = pick_from_range(*rng)
    return vstr(chosen) if chosen else None

def sanitize_source(src: str) -> str:
    # Remove SPDX lines (invalid IDs cause parser errors on newer solc)
    return SPDX_LINE_RE.sub("", src or "")

def detect_legacy_fallback(sources: Dict[str,str]) -> bool:
    # Pre-0.6 fallback syntax: `function () external` (with/without payable)
    for s in sources.values():
        if LEGACY_FALLBACK_RE.search(s):
            return True
    return False

def _collect_sources(project_dir: Path) -> Dict[str,str]:
    out: Dict[str,str] = {}
    for p in project_dir.rglob("*.sol"):
        try:
            out[p.relative_to(project_dir).as_posix()] = sanitize_source(p.read_text())
        except Exception:
            out[p.relative_to(project_dir).as_posix()] = ""
    return out

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if f.exists():
        try:
            raw = json.loads(f.read_text())
            # sanitize contents inside standard_input.json too
            if isinstance(raw, dict) and isinstance(raw.get("sources"), dict):
                raw = dict(raw)
                raw["sources"] = {k: {"content": sanitize_source(v.get("content") if isinstance(v, dict) else str(v))}
                                  for k, v in raw["sources"].items()}
            return raw
        except Exception:
            return None
    return None

def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    try:
        if _current_solc == version:
            return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed:
            install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _compile_standard_resilient(payload: dict, version: str):
    try:
        return compile_standard(payload)
    except Exception as e1:
        # Retry once without evmVersion if invalid
        text = (getattr(e1, "stderr_data", "") or "") + (getattr(e1, "stdout_data", "") or "") + str(e1)
        if "Invalid EVM version requested" in text:
            st = payload.get("settings") or {}
            if "evmVersion" in st:
                st2 = dict(st)
                st2.pop("evmVersion", None)
                p2 = dict(payload)
                p2["settings"] = st2
                return compile_standard(p2)
        raise

def _summarize(e: Exception, version: str, mode: str) -> str:
    et = type(e).__name__
    stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
    stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
    msg = f"{et} (v{version}, mode={mode})"
    if stderr or stdout:
        def tail(s): return s[-1200:] if isinstance(s, str) else ""
        msg += f"\n--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    else:
        msg += f": {e}"
    return msg

def is_reasonable_csv_version(v: str) -> bool:
    m = SEMVER3.search(v or "")
    if not m: return False
    M, mnr, p = map(int, m.groups())
    if M != 0 or mnr not in VALID_MINORS:
        return False
    maxp = LATEST_PATCH.get((M, mnr), None)
    if maxp is None: return False
    return 0 <= p <= maxp

def pick_initial_version(csv_version: str, sources: Dict[str,str]) -> str:
    # 1) CSV semver if present and reasonable
    if isinstance(csv_version, str) and csv_version.strip() and is_reasonable_csv_version(csv_version):
        return ".".join(SEMVER3.search(csv_version).groups())
    # 2) Intersect pragma ranges
    v = highest_pragma_version(sources)
    if v:
        return v
    # 3) default
    return "0.8.21"

def _compile_with(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err:
        return False, err, "init"

    # standard_input.json route
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = dict(std.get("settings") or {})
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {
                "language": "Solidity",
                "sources": std["sources"],  # already sanitized
                "settings": settings,
            }
            _ = _compile_standard_resilient(payload, version)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, _summarize(e, version, "standard"), "standard"

    # all-files route
    if sources:
        try:
            payload = {
                "language": "Solidity",
                "sources": {k: {"content": v} for k, v in sources.items()},  # sanitized
                "settings": {"optimizer": {"enabled": True, "runs": 200}},
            }
            _ = _compile_standard_resilient(payload, version)
            return True, "Compiled OK", "all_files"
        except Exception as e_all:
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                return False, _summarize(e_all, version, "all_files") + "\n---\n" + _summarize(e_single, version, "single_file"), "all_files"

    return False, "No .sol sources found", "none"

def _version_from_error_range(err_text: str, sources: Dict[str,str]) -> Optional[str]:
    m = ERR_LINE.search(err_text or "")
    if not m:
        return None
    lo, hi = parse_pragma_expr(m.group(1))
    chosen = pick_from_range(lo, hi)
    return vstr(chosen) if chosen else highest_pragma_version(sources)

def compile_project(project_dir: Path, csv_version: str) -> Tuple[bool,str,str,int,str]:
    std = _load_standard_input(project_dir)
    sources = None if std else _collect_sources(project_dir)
    n_sources = (len(std["sources"]) if std and "sources" in std else (len(sources) if sources else 0))

    src_for_infer = {k:(v.get("content") if isinstance(v, dict) else str(v)) for k,v in (std["sources"] if std else (sources or {})).items()} if (std or sources) else {}

    # Heuristic: legacy fallback syntax => prefer a 0.5.x right away if our initial pick would be >=0.6
    initial = pick_initial_version(csv_version, src_for_infer)
    if detect_legacy_fallback(src_for_infer):
        maj, minor, _ = vtup(initial)
        if (maj, minor) >= (0,6):
            initial = vstr(latest_in_minor(0,5))

    ok, msg, mode = _compile_with(initial, std, sources)
    if ok: return True, f"Compiled OK (initial v{initial}, mode={mode})", initial, n_sources, mode

    # Second try: strict best-from-pragmas (could differ from initial when CSV was ignored)
    pragma_best = highest_pragma_version(src_for_infer)
    if pragma_best and pragma_best != initial:
        ok2, msg2, mode2 = _compile_with(pragma_best, std, sources)
        if ok2: return True, f"Compiled OK (pragma_best v{pragma_best}, mode={mode2})", pragma_best, n_sources, mode2
        msg += "\n---\n" + msg2

    # Third: read pragma from error text and choose range-aware version
    hinted = _version_from_error_range(msg, src_for_infer)
    if hinted and hinted not in {initial, pragma_best}:
        ok3, msg3, mode3 = _compile_with(hinted, std, sources)
        if ok3: return True, f"Compiled OK (hinted v{hinted}, mode={mode3})", hinted, n_sources, mode3
        msg += "\n---\n" + msg3

    # Final fallback: if legacy fallback detected but we haven't tried 0.5.x yet
    if detect_legacy_fallback(src_for_infer):
        v05 = vstr(latest_in_minor(0,5))
        if v05 not in {initial, pragma_best, hinted}:
            ok4, msg4, mode4 = _compile_with(v05, std, sources)
            if ok4: return True, f"Compiled OK (legacy_fallback v{v05}, mode={mode4})", v05, n_sources, mode4
            msg += "\n---\n" + msg4

    return False, (msg or "Compilation failed (no further detail)"), initial, n_sources, "none"

# ---------------- main ------------------
df = pd.read_csv(INPUT_CSV)
todo = df.copy() if RECOMPILE_ALL else df.loc[df.get("compile_ok").astype(str).str.lower() != "true"].copy()

rows = []
it = range(len(todo))
if _HAS_TQDM:
    it = tqdm(it, desc="Recompiling (smart+heuristics)", unit="addr")

for i in it:
    row = todo.iloc[i]
    addr = str(row["contract_address"]).strip().lower()
    proj = PROJECTS_DIR / addr
    t0 = time.perf_counter()

    if not proj.exists():
        rows.append({
            "contract_address": addr,
            "project_dir": str(proj),
            "recompile_ok": False,
            "recompile_message": "Project directory not found",
            "used_version": "",
            "n_sources": 0,
            "mode": "none",
            "elapsed_s": round(time.perf_counter() - t0, 3),
            "original_idx": row.get("original_idx"),
            "label": row.get("label"),
        })
        continue

    csv_ver = str(row.get("compiler_version") or "").strip()
    ok, msg, used, nsrc, mode = compile_project(proj, csv_ver)

    rows.append({
        "contract_address": addr,
        "project_dir": str(proj),
        "recompile_ok": bool(ok),
        "recompile_message": str(msg),
        "used_version": used,
        "n_sources": int(nsrc),
        "mode": mode,
        "elapsed_s": round(time.perf_counter() - t0, 3),
        "original_idx": row.get("original_idx"),
        "label": row.get("label"),
    })

    if SLEEP_BETWEEN > 0: time.sleep(SLEEP_BETWEEN)

out = pd.DataFrame(rows)
out.to_csv(OUT_RECOMPILE, index=False)
print(f"[OK] Wrote: {OUT_RECOMPILE} ({len(out)} rows)")


Recompiling (smart+heuristics): 100%|██████████| 271/271 [02:13<00:00,  2.03addr/s]

[OK] Wrote: compilation_results_final.csv (271 rows)
